## GEO JSON DATASET MAKER PIPELINE
###### Optional: add .env of NCBI_API_KEY for faster fetching

In [ ]:
# %% [markdown]
# # GEO Metadata Pipeline (Final / Production)
#
# End-to-end, resumable pipeline that:
#
# 1. **Discover** — searches NCBI E-utilities for GSE (GEO Series) accessions published
#    within a configurable date range (explicit `start_date`/`end_date`, or `year`,
#    or `year`+`month`, or a rolling `years_back` window) and registers every discovered
#    GSE ID in a SQLite **registry table** (`status = "discovered"`).
# 2. **Ingest** — for a bounded batch of registry rows that are not yet `fetched`,
#    downloads full study + sample metadata via GEOparse and stores it in SQLite.
#    **No common/specific splitting** — every sample keeps its *entire* metadata
#    exactly as GEOparse returns it (per your decision: no duplication logic, no
#    data loss risk from merge bugs). Registry status becomes `fetched` on success,
#    `failed` (with an error message) on failure.
# 3. **Normalize** — reads every registry row that is `fetched` but not yet
#    `normalized` (this re-checks *all* such rows every run, not just the current
#    batch, so nothing is ever missed even if a previous run crashed mid-way) and
#    writes exactly two files per GSE: `{gse_id}.study.json` and `{gse_id}.sample.json`.
#
# The **SQLite registry is the single source of truth** for what has been discovered,
# fetched, and normalized. Re-running the pipeline is always safe and incremental.
#
# Run this file top-to-bottom in VS Code's Jupyter extension (each `# %%` block is a
# cell), or as a plain script: `python geo_metadata_pipeline_final.py`.

# %%
from __future__ import annotations

import calendar
import json
import logging
import os
import re
import sqlite3
from contextlib import contextmanager
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import requests
from GEOparse import get_GEO
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()
# --------------------------------------------------------------------------------------
# Logging
# --------------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("geo_pipeline")


# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
@dataclass(frozen=True)
class PipelineConfig:
    """Central configuration for every pipeline stage.

    Date-range resolution priority (see :func:`resolve_date_range`):
        1. ``start_date`` and ``end_date`` (explicit override, format ``YYYY/MM/DD``)
        2. ``year`` + ``month`` (a single calendar month)
        3. ``year`` only (the whole calendar year)
        4. ``years_back`` (rolling window ending today) — the default fallback

    Attributes:
        base_dir: Root working directory for all pipeline artifacts.
        db_path: Path to the SQLite database. Holds both the fetch/normalize
            registry and the raw ingested GSE data.
        semantic_study_dir: Output directory for ``{gse_id}.study.json`` files.
        semantic_sample_dir: Output directory for ``{gse_id}.sample.json`` files.
        year: Optional year filter (e.g. 2025).
        month: Optional month filter (1-12), only used together with ``year``.
        start_date: Optional explicit range start, ``YYYY/MM/DD``.
        end_date: Optional explicit range end, ``YYYY/MM/DD``.
        years_back: Fallback rolling window size in years if no other date
            option is set.
        fetch_limit: Max number of *new* (not-yet-fetched) GSE IDs to ingest
            in a single run. Controls how much work one run does.
        batch_size: Chunk size used for progress reporting / commit cadence
            while ingesting or normalizing. Does not limit total work, only
            how it's grouped.
        ncbi_request_delay_seconds: Delay between NCBI E-utilities requests.
            Automatically tightened when an API key is present.
        ncbi_timeout: Per-request HTTP timeout (seconds).
        max_retries: Retry attempts for transient NCBI / GEOparse failures.
        api_key: NCBI API key (raises rate limit from 3 req/s to 10 req/s).
            Defaults to the ``NCBI_API_KEY`` environment variable.
        email: Contact email sent to NCBI with every request (good citizenship /
            required for high-volume use). Defaults to ``NCBI_EMAIL``.
        geo_cache_dir: Directory GEOparse uses to cache downloaded SOFT files.
    """

    base_dir: Path = Path("geo_work")
    db_path: Path = field(default=None)  # type: ignore[assignment]
    semantic_study_dir: Path = field(default=None)  # type: ignore[assignment]
    semantic_sample_dir: Path = field(default=None)  # type: ignore[assignment]
    geo_cache_dir: Path = field(default=None)  # type: ignore[assignment]

    year: Optional[int] = None
    month: Optional[int] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    years_back: int = 5

    fetch_limit: int = 50
    batch_size: int = 25

    ncbi_request_delay_seconds: float = field(default=None)  # type: ignore[assignment]
    ncbi_timeout: int = 60
    max_retries: int = 4

    # Used as the `requests` timeout for the HTTPS SOFT-file download (see
    # _download_gse_soft_file). This is a *stall* timeout — no bytes received
    # for this many seconds — not a cap on total download time, so slow-but-
    # steady transfers (e.g. under concurrency, where bandwidth is shared)
    # are not cut off early. Kept separate from `max_retries` (NCBI API
    # retries) so one broken GSE never stalls a whole batch for too long.
    geo_fetch_timeout_seconds: int = 60
    geo_fetch_max_retries: int = 2
    # Fetching GSEs in parallel means one hung/broken GSE no longer blocks the
    # rest of the batch behind it. NCBI's FTP/HTTP mirror comfortably handles
    # a handful of concurrent downloads; keep this modest and polite.
    max_concurrent_fetches: int = 4

    # If True, skip the discovery/registration NCBI calls entirely and just
    # keep working through whatever is already in the registry (fetch pending
    # `discovered`/`failed` rows, normalize pending `fetched`/`normalize_failed`
    # rows). Useful once you've already discovered a date range's GSE IDs and
    # just want to keep chipping away at fetch_limit-sized batches without
    # burning an NCBI esearch/esummary round-trip on every single run.
    skip_discovery: bool = False

    api_key: Optional[str] = field(default_factory=lambda: os.getenv("NCBI_API_KEY"))
    email: Optional[str] = field(default_factory=lambda: os.getenv("NCBI_EMAIL"))

    def __post_init__(self) -> None:
        # Derive dependent paths / adaptive delay if not explicitly provided.
        object.__setattr__(self, "db_path", self.db_path or self.base_dir / "geo_data.db")
        object.__setattr__(
            self, "semantic_study_dir", self.semantic_study_dir or self.base_dir / "studies"
        )
        object.__setattr__(
            self, "semantic_sample_dir", self.semantic_sample_dir or self.base_dir / "samples"
        )
        object.__setattr__(
            self, "geo_cache_dir", self.geo_cache_dir or self.base_dir / "geo_cache"
        )
        if self.ncbi_request_delay_seconds is None:
            # NCBI allows 10 req/s with an API key, 3 req/s without.
            object.__setattr__(
                self, "ncbi_request_delay_seconds", 0.11 if self.api_key else 0.34
            )

    def ensure_dirs(self) -> None:
        """Create every directory this config references, if missing."""
        for d in (
            self.base_dir,
            self.semantic_study_dir,
            self.semantic_sample_dir,
            self.geo_cache_dir,
        ):
            d.mkdir(parents=True, exist_ok=True)


def resolve_date_range(cfg: PipelineConfig) -> Tuple[str, str]:
    """Resolve the effective (start_date, end_date) for GSE discovery.

    Priority: explicit start/end > year+month > year only > rolling years_back.

    Args:
        cfg: The active pipeline configuration.

    Returns:
        A ``(start_date, end_date)`` tuple, each formatted ``YYYY/MM/DD``.
    """
    if cfg.start_date and cfg.end_date:
        return cfg.start_date, cfg.end_date

    if cfg.year and cfg.month:
        last_day = calendar.monthrange(cfg.year, cfg.month)[1]
        return f"{cfg.year}/{cfg.month:02d}/01", f"{cfg.year}/{cfg.month:02d}/{last_day:02d}"

    if cfg.year:
        return f"{cfg.year}/01/01", f"{cfg.year}/12/31"

    today = date.today()
    try:
        start = today.replace(year=today.year - cfg.years_back)
    except ValueError:
        # Handles Feb 29 on non-leap target years.
        start = today.replace(year=today.year - cfg.years_back, day=today.day - 1)
    return start.strftime("%Y/%m/%d"), today.strftime("%Y/%m/%d")


# %% [markdown]
# ## Shared HTTP helper — retrying NCBI E-utilities client
#
# Centralizes timeout, retry/backoff, and API-key/email injection for every NCBI call.
# Retries only on transient failures (timeouts, connection errors, 5xx); a 4xx response
# fails fast since retrying it can never succeed.

# %%
class PipelineError(Exception):
    """Raised for unrecoverable pipeline failures (e.g. exhausted NCBI retries)."""


_RETRYABLE_STATUS = {429, 500, 502, 503, 504}


def ncbi_get(cfg: PipelineConfig, url: str, params: Dict[str, Any]) -> requests.Response:
    """GET an NCBI E-utilities endpoint with retry/backoff and rate-limit courtesy.

    Args:
        cfg: The active pipeline configuration (supplies timeout, api_key, email,
            delay, and retry count).
        url: Full endpoint URL (e.g. the esearch or esummary URL).
        params: Query parameters for the request.

    Returns:
        The successful ``requests.Response``.

    Raises:
        PipelineError: If every retry attempt fails, or a non-retryable HTTP
            error (e.g. 400 bad request) is returned.
    """
    params = dict(params)
    if cfg.api_key:
        params.setdefault("api_key", cfg.api_key)
    if cfg.email:
        params.setdefault("email", cfg.email)

    last_error: Optional[Exception] = None
    for attempt in range(cfg.max_retries):
        try:
            response = requests.get(url, params=params, timeout=cfg.ncbi_timeout)
            if response.status_code in _RETRYABLE_STATUS:
                raise requests.HTTPError(f"Retryable status {response.status_code}")
            response.raise_for_status()
            time.sleep(cfg.ncbi_request_delay_seconds)
            return response
        except requests.HTTPError as exc:
            # Non-retryable 4xx (other than 429, already filtered above) -> fail fast.
            status = getattr(exc.response, "status_code", None)
            if status is not None and status not in _RETRYABLE_STATUS and 400 <= status < 500:
                raise PipelineError(f"Non-retryable NCBI error {status} for {url}: {exc}") from exc
            last_error = exc
        except (requests.ConnectionError, requests.Timeout) as exc:
            last_error = exc

        backoff = min(2 ** attempt, 8)
        logger.warning("NCBI request failed (attempt %d/%d): %s — retrying in %ss",
                        attempt + 1, cfg.max_retries, last_error, backoff)
        time.sleep(backoff)

    raise PipelineError(f"NCBI request permanently failed for {url}: {last_error}")


def _gse_ftp_directory(gse_id: str) -> str:
    """Compute the NCBI GEO FTP/HTTPS parent directory name for a GSE accession.

    NCBI groups GSE series into directories by replacing the last 3 digits of
    the accession number with ``nnn`` (e.g. ``GSE336978`` -> ``GSE336nnn``;
    ``GSE42`` -> ``GSEnnn``). This is NCBI's documented, stable directory
    convention for the ``geo/series/`` tree.

    Args:
        gse_id: A GSE accession, e.g. ``"GSE336978"``.

    Returns:
        The directory name, e.g. ``"GSE336nnn"``.
    """
    number = int(gse_id[3:])
    thousands = number // 1000
    return f"GSE{thousands}nnn" if thousands else "GSEnnn"


def _build_soft_gz_url(gse_id: str) -> str:
    """Build the HTTPS URL for a GSE's ``_family.soft.gz`` file.

    NCBI mirrors its FTP tree over HTTPS at the same host/paths, which is
    generally far more reliable than raw FTP on networks with firewalls,
    proxies, or antivirus software that interferes with FTP/passive-mode
    connections (a very common source of the misleading "downloaded size
    does not match" error GEOparse's own FTP client reports).

    Args:
        gse_id: A GSE accession, e.g. ``"GSE336978"``.

    Returns:
        The full HTTPS URL to the series' SOFT file.
    """
    directory = _gse_ftp_directory(gse_id)
    return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{directory}/{gse_id}/soft/{gse_id}_family.soft.gz"


def _download_gse_soft_file(cfg: PipelineConfig, gse_id: str) -> Path:
    """Download a GSE's SOFT file over HTTPS, verifying it arrived complete.

    Streams the download to a ``.part`` temp file and, if the server reports
    a ``Content-Length``, compares it against the actual bytes written —
    catching a truncated/interrupted download immediately with an explicit,
    unambiguous error, rather than silently handing GEOparse a corrupt file
    (which produces the confusing "ID could be incorrect or data might not
    be public yet" message even when the real cause was a timeout or a
    dropped connection). Only renamed to the final path once verified.

    Args:
        cfg: The active pipeline configuration (supplies the cache dir and
            per-request timeout).
        gse_id: The GSE accession to download.

    Returns:
        The path to the verified, fully-downloaded ``.soft.gz`` file.

    Raises:
        requests.HTTPError: On a non-2xx HTTP response.
        PipelineError: If the downloaded byte count doesn't match the
            server-reported ``Content-Length``.
    """
    url = _build_soft_gz_url(gse_id)
    cfg.geo_cache_dir.mkdir(parents=True, exist_ok=True)
    final_path = cfg.geo_cache_dir / f"{gse_id}_family.soft.gz"
    tmp_path = final_path.with_suffix(final_path.suffix + ".part")

    with requests.get(url, stream=True, timeout=cfg.geo_fetch_timeout_seconds) as response:
        response.raise_for_status()
        expected_size_header = response.headers.get("Content-Length")
        expected_size = int(expected_size_header) if expected_size_header else None

        written = 0
        with open(tmp_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 64):
                if chunk:
                    f.write(chunk)
                    written += len(chunk)

    if expected_size is not None and written != expected_size:
        tmp_path.unlink(missing_ok=True)
        raise PipelineError(
            f"Incomplete download for {gse_id}: got {written} bytes, "
            f"expected {expected_size} (connection likely dropped or timed out)."
        )

    tmp_path.replace(final_path)
    return final_path


def get_geo_with_retry(cfg: PipelineConfig, gse_id: str) -> Any:
    """Fetch and parse a GSE, downloading over HTTPS (not GEOparse's built-in FTP).

    GEOparse's own FTP downloader has no real per-request timeout and, on
    networks where FTP is throttled or interfered with (firewalls, proxies,
    antivirus — very common on corporate/home Windows setups), tends to
    return truncated files that GEOparse then misreports as "ID could be
    incorrect or data might not be public yet". Downloading over HTTPS with
    :func:`_download_gse_soft_file` (which has an explicit timeout and
    verifies the download size) avoids that failure mode entirely; GEOparse
    is then only used to *parse* the already-downloaded local file.

    Args:
        cfg: The active pipeline configuration.
        gse_id: The GSE accession to fetch.

    Returns:
        A GEOparse ``GSE`` object.

    Raises:
        PipelineError: If every retry attempt fails.
    """
    last_error: Optional[Exception] = None
    for attempt in range(cfg.geo_fetch_max_retries):
        try:
            soft_path = _download_gse_soft_file(cfg, gse_id)
            return get_GEO(filepath=str(soft_path), silent=True)
        except Exception as exc:  # noqa: BLE001 - network + GEOparse parse errors both land here
            last_error = exc
            backoff = min(2 ** attempt, 8)
            logger.warning(
                "GEOparse fetch failed for %s (attempt %d/%d): %s — retrying in %ss",
                gse_id, attempt + 1, cfg.geo_fetch_max_retries, exc, backoff,
            )
            time.sleep(backoff)

    raise PipelineError(f"GEOparse fetch permanently failed for {gse_id}: {last_error}")


# %% [markdown]
# ## SQLite schema — registry (source of truth) + raw data
#
# `gse_registry` tracks the lifecycle of every discovered GSE ID:
# `discovered -> fetched -> normalized`, or `failed` (with an error message) at
# either the fetch or normalize step. `gse_data` holds the raw ingested metadata.
#
# Every run re-checks the *entire* registry for rows stuck at `fetched` (or
# previously `failed`) so a crash never permanently loses a GSE.

# %%
# Valid registry statuses, enforced in application code (mark_status) rather
# than a SQLite CHECK constraint — CHECK constraints on an existing table
# can't be altered without recreating it, which makes adding a new status
# later (like `normalize_failed`, added here) a painful migration. A plain
# TEXT column plus a Python-side guard gives the same safety without that cost.
VALID_STATUSES = {"discovered", "fetched", "normalized", "failed", "normalize_failed"}


@contextmanager
def db_connection(db_path: Path):
    """Open a SQLite connection with production-safe pragmas, and always close it.

    Applies ``WAL`` journal mode (so concurrent readers/writers don't block
    each other and don't corrupt on a crash), a ``busy_timeout`` (so a
    momentarily locked database retries instead of raising immediately), and
    ``foreign_keys = ON`` (SQLite does not enforce declared foreign keys
    unless this pragma is set on every connection). Commits on clean exit,
    rolls back on exception, and always closes the connection afterward —
    unlike bare ``sqlite3.connect(...) as conn``, which commits/rolls back
    but leaves the connection open for garbage collection to clean up.

    Args:
        db_path: Path to the SQLite database file.

    Yields:
        An open ``sqlite3.Connection``.
    """
    conn = sqlite3.connect(db_path, timeout=30)
    try:
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA busy_timeout=30000;")
        conn.execute("PRAGMA foreign_keys=ON;")
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def init_db(db_path: Path) -> None:
    """Create the registry and data tables if they do not already exist.

    Args:
        db_path: Path to the SQLite database file.
    """
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with db_connection(db_path) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS gse_registry (
                gse_id TEXT PRIMARY KEY,
                status TEXT NOT NULL,
                discovered_at TEXT,
                fetched_at TEXT,
                normalized_at TEXT,
                error TEXT
            );
            """
        )
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS gse_data (
                gse_id TEXT PRIMARY KEY,
                title TEXT,
                organism TEXT,
                summary TEXT,
                sample_records TEXT,
                supplementary_files TEXT,
                platform_id TEXT,
                metadata_json TEXT,
                FOREIGN KEY (gse_id) REFERENCES gse_registry (gse_id)
            );
            """
        )
        conn.execute("CREATE INDEX IF NOT EXISTS idx_registry_status ON gse_registry(status);")
    logger.info("Database ready at %s", db_path)



def _now() -> str:
    """Return the current UTC-local timestamp as an ISO-ish string."""
    return time.strftime("%Y-%m-%d %H:%M:%S")


def register_discovered_ids(gse_ids: List[str], db_path: Path) -> int:
    """Insert newly discovered GSE IDs into the registry as ``discovered``.

    IDs already present (in any status) are left untouched — this never
    downgrades a ``fetched``/``normalized`` row back to ``discovered``.

    Args:
        gse_ids: GSE accessions discovered in this run.
        db_path: Path to the SQLite database.

    Returns:
        The number of genuinely new rows inserted.
    """
    init_db(db_path)
    now = _now()
    with db_connection(db_path) as conn:
        before = conn.execute("SELECT COUNT(*) FROM gse_registry;").fetchone()[0]
        conn.executemany(
            "INSERT OR IGNORE INTO gse_registry (gse_id, status, discovered_at) VALUES (?, 'discovered', ?);",
            [(gse_id, now) for gse_id in gse_ids],
        )
        after = conn.execute("SELECT COUNT(*) FROM gse_registry;").fetchone()[0]
    inserted = after - before
    logger.info("Registered %d new GSE ID(s) (of %d discovered).", inserted, len(gse_ids))
    return inserted


def select_ids_to_fetch(db_path: Path, fetch_limit: int) -> List[str]:
    """Select up to ``fetch_limit`` registry rows needing ingestion.

    Includes both never-tried (``discovered``) and previously fetch-failed
    (``failed``) rows, so transient fetch failures are retried automatically
    on the next run. Deliberately excludes ``normalize_failed`` rows — those
    already fetched successfully and only need re-normalizing, not a full
    re-download (see :func:`select_ids_to_normalize`).

    Args:
        db_path: Path to the SQLite database.
        fetch_limit: Maximum number of GSE IDs to select.

    Returns:
        A list of GSE accession strings to ingest next.
    """
    with db_connection(db_path) as conn:
        rows = conn.execute(
            """
            SELECT gse_id FROM gse_registry
            WHERE status IN ('discovered', 'failed')
            ORDER BY discovered_at ASC
            LIMIT ?;
            """,
            (fetch_limit,),
        ).fetchall()
    return [r[0] for r in rows]


def select_ids_to_normalize(db_path: Path) -> List[str]:
    """Select every registry row ready to (re-)normalize.

    Includes rows that are ``fetched`` (never normalized yet) as well as
    ``normalize_failed`` (fetch succeeded, but a previous normalize attempt
    raised) — the latter is a distinct status from a fetch failure (see
    :func:`mark_status`), so retrying it never triggers a wasted re-download.

    This intentionally scans the *whole* registry (not just the current
    batch) every run, so a GSE that was ingested but never normalized
    (e.g. the process crashed between stages) is always picked back up.

    Args:
        db_path: Path to the SQLite database.

    Returns:
        A list of GSE accession strings ready to normalize.
    """
    with db_connection(db_path) as conn:
        rows = conn.execute(
            "SELECT gse_id FROM gse_registry WHERE status IN ('fetched', 'normalize_failed');"
        ).fetchall()
    return [r[0] for r in rows]


def _apply_status_update(
    conn: sqlite3.Connection,
    gse_id: str,
    status: str,
    timestamp_column: Optional[str] = None,
    error: Optional[str] = None,
) -> None:
    """Apply a registry status update using an already-open connection.

    Factored out of :func:`mark_status` so callers that already hold an open
    connection for a batch of work (e.g. :func:`ingest_gse_batch`) can reuse
    it instead of opening a second connection per row.

    Args:
        conn: An open SQLite connection (caller owns commit/close).
        gse_id: The GSE accession to update.
        status: New status value — must be one of :data:`VALID_STATUSES`.
        timestamp_column: If given, also stamp this column with the current time.
        error: Optional error message to store.

    Raises:
        ValueError: If ``status`` is not one of :data:`VALID_STATUSES`.
    """
    if status not in VALID_STATUSES:
        raise ValueError(f"Invalid registry status {status!r}; must be one of {VALID_STATUSES}")

    if timestamp_column:
        conn.execute(
            f"UPDATE gse_registry SET status = ?, error = ?, {timestamp_column} = ? WHERE gse_id = ?;",
            (status, error, _now(), gse_id),
        )
    else:
        conn.execute(
            "UPDATE gse_registry SET status = ?, error = ? WHERE gse_id = ?;",
            (status, error, gse_id),
        )


def mark_status(
    db_path: Path,
    gse_id: str,
    status: str,
    timestamp_column: Optional[str] = None,
    error: Optional[str] = None,
) -> None:
    """Update a registry row's status (and optional timestamp / error).

    Opens and closes its own connection — use :func:`_apply_status_update`
    directly instead if you already hold one open for a batch.

    Args:
        db_path: Path to the SQLite database.
        gse_id: The GSE accession to update.
        status: New status value — one of :data:`VALID_STATUSES`. Use
            ``"failed"`` for a fetch-stage failure (triggers re-fetch) and
            ``"normalize_failed"`` for a normalize-stage failure (retries
            only normalization, not the download) — see :func:`select_ids_to_fetch`
            vs :func:`select_ids_to_normalize`.
        timestamp_column: If given, also stamp this column (e.g.
            ``"fetched_at"``) with the current time.
        error: Optional error message to store (used for failure statuses).

    Raises:
        ValueError: If ``status`` is not one of :data:`VALID_STATUSES`.
    """
    with db_connection(db_path) as conn:
        _apply_status_update(conn, gse_id, status, timestamp_column, error)


def registry_summary(db_path: Path) -> Dict[str, int]:
    """Return a count of registry rows grouped by status (for reporting).

    Args:
        db_path: Path to the SQLite database.

    Returns:
        A dict mapping status -> row count.
    """
    with db_connection(db_path) as conn:
        rows = conn.execute("SELECT status, COUNT(*) FROM gse_registry GROUP BY status;").fetchall()
    return dict(rows)


# %% [markdown]
# ## Stage 1 — Date-filtered GSE discovery

# %%
GEO_ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
GEO_ESUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
NCBI_RETMAX = 200


def discover_gse_ids(cfg: PipelineConfig) -> List[str]:
    """Discover every GSE accession published within the configured date range.

    Args:
        cfg: The active pipeline configuration.

    Returns:
        A de-duplicated, order-preserving list of GSE accession strings.
    """
    start_date, end_date = resolve_date_range(cfg)
    date_query = f'("{start_date}"[PDAT] : "{end_date}"[PDAT]) AND "gse"[Filter]'
    logger.info("Discovering GSE accessions: %s to %s", start_date, end_date)

    gse_ids: List[str] = []
    seen: set[str] = set()
    retstart = 0

    while True:
        search_params = {
            "db": "gds", "term": date_query, "retmode": "json",
            "retmax": NCBI_RETMAX, "retstart": retstart,
        }
        response = ncbi_get(cfg, GEO_ESEARCH_URL, search_params)
        idlist = response.json().get("esearchresult", {}).get("idlist", [])
        if not idlist:
            break

        summary_response = ncbi_get(
            cfg, GEO_ESUMMARY_URL, {"db": "gds", "id": ",".join(idlist), "retmode": "json"}
        )
        summary = summary_response.json().get("result", {})

        for uid in idlist:
            record = summary.get(uid, {})
            accession = record.get("accession", "")
            entry_type = str(record.get("entrytype", "")).lower()
            if entry_type == "gse" and accession.startswith("GSE") and accession not in seen:
                gse_ids.append(accession)
                seen.add(accession)

        logger.info("Discovered %d GSE ID(s) so far (retstart=%d)", len(gse_ids), retstart)
        retstart += NCBI_RETMAX

    return gse_ids


# %% [markdown]
# ## Stage 2 — Ingestion (GEOparse -> SQLite), full per-sample metadata, no splitting
#
# Every sample keeps its complete metadata as returned by GEOparse — no common/specific
# splitting, no merge-back step, nothing to lose track of. Storage cost is a few extra
# duplicated keys per sample, which is a deliberate, accepted trade-off for simplicity
# and correctness.

# %%
_EXCLUDED_STUDY_TOP_LEVEL_KEYS = {
    "title", "organism", "summary", "sample_id",
    "supplementary_file", "platform_id", "geo_accession",
}


def _safe_get(metadata: Dict[str, Any], key: str, default: Any = None) -> Any:
    """Return the first element of a GEOparse metadata list field, or a default.

    Args:
        metadata: A GEOparse ``.metadata`` dict.
        key: The metadata key to look up.
        default: Value to return if the key is missing or empty.

    Returns:
        The first list element, the raw scalar value, or ``default``.
    """
    value = metadata.get(key, [default])
    return value[0] if isinstance(value, list) else value


def _get_organism(gse: Any) -> Optional[str]:
    """Resolve the organism(s) for a GSE, falling back to per-sample organisms.

    Args:
        gse: A GEOparse ``GSE`` object.

    Returns:
        A semicolon-joined, sorted, de-duplicated organism string, or ``None``.
    """
    if "organism" in gse.metadata:
        organisms = gse.metadata["organism"]
        return ";".join(sorted(set(organisms))) if isinstance(organisms, list) else organisms

    organisms: set[str] = set()
    for gsm in gse.gsms.values():
        value = gsm.metadata.get("organism_ch1")
        if value:
            organisms.update(value if isinstance(value, list) else [value])
    return ";".join(sorted(organisms)) if organisms else None


def _extract_bioproject(gse: Any) -> Optional[str]:
    """Extract a clean PRJNA accession from a GSE's ``relation`` metadata, if present.

    This only parses metadata GEOparse already fetched — it makes no extra
    network call (SRA/BioProject run-level fetching was deliberately dropped).

    Args:
        gse: A GEOparse ``GSE`` object.

    Returns:
        The PRJNA accession string, or ``None`` if not found.
    """
    for relation in gse.metadata.get("relation", []):
        match = re.search(r"(PRJNA\d+)", str(relation))
        if match:
            return match.group(1)
    return None


def _extract_series_relationships(gse: Any) -> Tuple[List[str], Optional[str]]:
    """Extract SuperSeries/SubSeries links from a GSE's ``relation`` metadata.

    GEO SuperSeries records list their children as ``"SuperSeries of: GSExxxxx"``
    relation entries; SubSeries records list their parent as
    ``"SubSeries of: GSExxxxx"``. A SuperSeries carries no sample data itself —
    the real study/sample content lives in its SubSeries.

    Args:
        gse: A GEOparse ``GSE`` object.

    Returns:
        A ``(subseries_ids, parent_series_id)`` tuple:
        ``subseries_ids`` is the list of child GSE accessions if this record is
        a SuperSeries (empty otherwise); ``parent_series_id`` is the parent GSE
        accession if this record is itself a SubSeries (``None`` otherwise).
    """
    subseries_ids: List[str] = []
    parent_series_id: Optional[str] = None

    for relation in gse.metadata.get("relation", []):
        relation = str(relation)
        child_match = re.search(r"SuperSeries of:\s*(GSE\d+)", relation, re.IGNORECASE)
        if child_match:
            subseries_ids.append(child_match.group(1))
            continue
        parent_match = re.search(r"SubSeries of:\s*(GSE\d+)", relation, re.IGNORECASE)
        if parent_match:
            parent_series_id = parent_match.group(1)

    return subseries_ids, parent_series_id


# Key-level (not substring) classification: matched against the part of a
# "key: value" characteristic string BEFORE the colon, normalized to lowercase.
# This fixes the earlier bug where `"age" in "developmental stage"` false-matched.
_TISSUE_KEYS = {"tissue", "tissue type", "source tissue"}
_TIMEPOINT_KEYS = {"age", "time", "timepoint", "time point"}
_STRAIN_KEYS = {"strain", "genetic background"}


def _classify_characteristic_key(key: str) -> Optional[str]:
    """Classify a characteristic's key (text before the colon) exactly.

    Args:
        key: The raw key text preceding the colon in a
            ``characteristics_ch1`` entry (e.g. ``"developmental stage"``).

    Returns:
        ``"tissue"``, ``"timepoint"``, ``"strain"``, or ``None`` if the key
        doesn't exactly match a known set (whole-key match, never a
        substring match, so e.g. "stage" never matches the "age" key).
    """
    key_norm = key.strip().lower()
    if key_norm in _TISSUE_KEYS:
        return "tissue"
    if key_norm in _TIMEPOINT_KEYS:
        return "timepoint"
    if key_norm in _STRAIN_KEYS:
        return "strain"
    return None


def _build_sample_records(gse: Any) -> List[Dict[str, Any]]:
    """Build one complete metadata record per GSM (sample) in a GSE.

    Every field GEOparse returns for the sample is preserved verbatim under
    ``extra_metadata``; a handful of convenience fields (tissue, timepoint,
    strain) are additionally lifted to the top level using exact key
    classification (see :func:`_classify_characteristic_key`).

    Args:
        gse: A GEOparse ``GSE`` object.

    Returns:
        A list of per-sample metadata dictionaries, each fully self-contained.
    """
    samples = []
    for gsm_id, gsm in gse.gsms.items():
        tissue = timepoint = strain = None
        for characteristic in gsm.metadata.get("characteristics_ch1", []):
            if ":" not in characteristic:
                continue
            key_part, value_part = characteristic.split(":", 1)
            category = _classify_characteristic_key(key_part)
            value = value_part.strip()
            if category == "tissue":
                tissue = value
            elif category == "timepoint":
                timepoint = value
            elif category == "strain":
                strain = value

        samples.append(
            {
                "gsm_id": gsm_id,
                "title": _safe_get(gsm.metadata, "title"),
                "source_name": _safe_get(gsm.metadata, "source_name_ch1"),
                "organism": _safe_get(gsm.metadata, "organism_ch1"),
                "tissue": tissue,
                "timepoint": timepoint,
                "strain": strain,
                "platform_id": _safe_get(gsm.metadata, "platform_id"),
                "instrument_model": _safe_get(gsm.metadata, "instrument_model"),
                "library_strategy": _safe_get(gsm.metadata, "library_strategy"),
                "extra_metadata": dict(gsm.metadata),
            }
        )
    return samples


def _fetch_gse_payload(cfg: PipelineConfig, gse_id: str) -> Dict[str, Any]:
    """Fetch one GSE and build its full in-memory payload (network-only, no DB access).

    Runs safely inside a worker thread — touches only the network and the
    local GEOparse cache directory, never the SQLite connection.

    Args:
        cfg: The active pipeline configuration.
        gse_id: The GSE accession to fetch.

    Returns:
        A dict ready to be written to ``gse_data`` / used to update the registry.
    """
    gse = get_geo_with_retry(cfg, gse_id)

    sample_records = _build_sample_records(gse)
    bioproject = _extract_bioproject(gse)
    subseries_ids, parent_series_id = _extract_series_relationships(gse)

    extra_metadata = {
        k: v for k, v in gse.metadata.items() if k not in _EXCLUDED_STUDY_TOP_LEVEL_KEYS
    }
    if bioproject:
        extra_metadata["bioproject"] = bioproject
    if subseries_ids:
        extra_metadata["subseries_of_this_superseries"] = subseries_ids
    if parent_series_id:
        extra_metadata["parent_series_id"] = parent_series_id

    return {
        "gse_id": gse_id,
        "title": _safe_get(gse.metadata, "title"),
        "organism": _get_organism(gse),
        "summary": _safe_get(gse.metadata, "summary"),
        "sample_records": sample_records,
        "supplementary_files": gse.metadata.get("supplementary_file", []),
        "platform_id": ",".join(gse.metadata.get("platform_id", [])),
        "extra_metadata": extra_metadata,
        "subseries_ids": subseries_ids,
    }


def ingest_gse_batch(gse_ids: Iterable[str], cfg: PipelineConfig) -> Tuple[int, int]:
    """Fetch and store full study + sample metadata for a batch of GSE IDs.

    GSEs are fetched **concurrently** (up to ``cfg.max_concurrent_fetches`` at
    once) so a single hung or broken GSE download no longer stalls every GSE
    queued behind it — it only occupies one worker slot while the rest of the
    batch keeps moving. Each fetch uses its own per-request timeout via
    ``requests`` (see :func:`_download_gse_soft_file`) rather than a shared
    global timeout, so this is safe to run across threads with no risk of a
    global-state race. SQLite writes happen sequentially on the main thread
    as each fetch completes, so the database is never touched from a worker
    thread. Updates the registry to ``fetched`` on success or ``failed`` (with
    the error message) on failure; one bad GSE never blocks the batch.

    Args:
        gse_ids: The batch of GSE accession strings to ingest.
        cfg: The active pipeline configuration.

    Returns:
        A ``(succeeded, failed)`` count tuple.
    """
    gse_ids = list(gse_ids)
    succeeded = failed = 0

    with db_connection(cfg.db_path) as conn, \
         ThreadPoolExecutor(max_workers=cfg.max_concurrent_fetches) as pool:

        future_to_id = {pool.submit(_fetch_gse_payload, cfg, gid): gid for gid in gse_ids}

        for future in tqdm(as_completed(future_to_id), total=len(future_to_id), desc="Ingesting GSE batch"):
            gse_id = future_to_id[future]
            try:
                payload = future.result()

                if payload["subseries_ids"]:
                    # This GSE is a SuperSeries with no sample data of its own —
                    # register its children so they get fetched on a future batch.
                    now = _now()
                    conn.executemany(
                        "INSERT OR IGNORE INTO gse_registry (gse_id, status, discovered_at) VALUES (?, 'discovered', ?);",
                        [(sub_id, now) for sub_id in payload["subseries_ids"]],
                    )
                    logger.info(
                        "%s is a SuperSeries; registered %d SubSeries ID(s) for future fetch: %s",
                        gse_id, len(payload["subseries_ids"]), ", ".join(payload["subseries_ids"]),
                    )

                conn.execute(
                    """
                    INSERT OR REPLACE INTO gse_data
                    (gse_id, title, organism, summary, sample_records,
                     supplementary_files, platform_id, metadata_json)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        payload["gse_id"],
                        payload["title"],
                        payload["organism"],
                        payload["summary"],
                        json.dumps(payload["sample_records"]),
                        json.dumps(payload["supplementary_files"]),
                        payload["platform_id"],
                        json.dumps(payload["extra_metadata"]),
                    ),
                )
                # Reuse the batch's single open connection for the status
                # update too, rather than mark_status() opening a second
                # connection per row — avoids open/close churn while this
                # connection is already held for the whole batch.
                _apply_status_update(conn, gse_id, "fetched", timestamp_column="fetched_at")
                conn.commit()
                succeeded += 1
                logger.info("Ingested %s: %d sample(s).", gse_id, len(payload["sample_records"]))
            except Exception as exc:  # noqa: BLE001
                logger.exception("Failed to ingest %s; continuing with next GSE.", gse_id)
                _apply_status_update(conn, gse_id, "failed", error=str(exc))
                conn.commit()
                failed += 1

    logger.info("Ingest batch complete: %d succeeded, %d failed (of %d).", succeeded, failed, len(gse_ids))
    return succeeded, failed


# %% [markdown]
# ## Stage 3 — Normalize directly from the database into final study/sample JSON

# %%
_DB_JSON_COLUMNS = ("sample_records", "supplementary_files", "metadata_json")


def get_gse_record_from_db(gse_id: str, db_path: Path) -> Optional[Dict[str, Any]]:
    """Load a single GSE's stored record from the database as a plain dict.

    Args:
        gse_id: The GSE accession to load.
        db_path: Path to the SQLite database.

    Returns:
        A dict with JSON columns already parsed, or ``None`` if not found.
    """
    with db_connection(db_path) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute("SELECT * FROM gse_data WHERE gse_id = ?;", (gse_id,)).fetchone()

    if row is None:
        return None

    record = dict(row)
    for column in _DB_JSON_COLUMNS:
        if record.get(column):
            record[column] = json.loads(record[column])
    return record


def _atomic_write_json(path: Path, data: Any) -> None:
    """Write ``data`` as indented JSON to ``path`` atomically (tmp file + rename).

    Args:
        path: Destination file path.
        data: JSON-serializable data to write.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp_path.replace(path)


_BIO_CHARACTERISTIC_KEYS = {
    "tissue", "cell line", "cell type", "genotype", "specimen", "disease", "organism", "strain",
}
# "time"/"timepoint" are deliberately excluded: GEO's "age" characteristic is
# the one surfaced in experimental_conditions (see generate_sample_semantic).
# A characteristic literally keyed "time" (distinct from age) now falls
# through to `other`/raw_extra_metadata instead of a conditions bucket that
# nothing reads — preserving the data rather than silently dropping it.
_CONDITION_CHARACTERISTIC_KEYS = {"treatment", "age", "stage", "exposure"}


def _first(value: Any) -> Any:
    """Return the first element of a list, or ``value`` unchanged if not a list."""
    return value[0] if isinstance(value, list) and value else value


def _join(value: Any, sep: str = "; ") -> Any:
    """Join a list of values into a single string; pass non-lists through unchanged."""
    if isinstance(value, list):
        return sep.join(str(v) for v in value if v)
    return value


def _normalize_key(key: str) -> str:
    """Lowercase and underscore-format a metadata key for consistent output."""
    return key.strip().lower().replace(" ", "_")


def _clean(obj: Any) -> Any:
    """Recursively drop ``None``/empty-string/empty-list/empty-dict values.

    Args:
        obj: A (possibly nested) dict, list, or scalar.

    Returns:
        The same structure with empty/null values removed.
    """
    if isinstance(obj, dict):
        cleaned = {k: _clean(v) for k, v in obj.items()}
        return {k: v for k, v in cleaned.items() if v not in (None, "", [], {})}
    if isinstance(obj, list):
        cleaned_list = [_clean(v) for v in obj]
        return [v for v in cleaned_list if v not in (None, "", [], {})]
    return obj


def _representative_sample_metadata(record: Dict[str, Any]) -> Dict[str, Any]:
    """Return a representative sample's ``extra_metadata`` for study-level fields.

    Sequencing/protocol fields (instrument model, library strategy, growth/
    treatment/extraction protocol, data processing, etc.) are GSM (sample)
    -level fields in GEOparse — they never appear on ``gse.metadata`` itself.
    Since common/specific splitting was intentionally removed (every sample
    keeps its full metadata), the first sample is used as a stand-in for the
    study as a whole; these fields are expected to be identical or
    near-identical across every sample in a given study.

    Args:
        record: A GSE record as returned by :func:`get_gse_record_from_db`.

    Returns:
        The first sample's ``extra_metadata`` dict, or ``{}`` if the GSE has
        no stored samples (e.g. an empty SuperSeries — see
        :func:`_extract_series_relationships`).
    """
    samples = record.get("sample_records") or []
    if not samples:
        return {}
    return samples[0].get("extra_metadata", {}) or {}


def generate_study_semantic(record: Dict[str, Any]) -> Dict[str, Any]:
    """Convert a GSE database record into a clean, semantic study-level document.

    Args:
        record: A GSE record as returned by :func:`get_gse_record_from_db`.

    Returns:
        A semantic study JSON document (nulls/empties removed).
    """
    metadata = record.get("metadata_json") or {}
    samples = record.get("sample_records") or []
    sample_meta = _representative_sample_metadata(record)

    study_json = {
        "gse_id": record.get("gse_id"),
        "title": record.get("title"),
        "organism": record.get("organism"),
        "study_summary": record.get("summary"),
        "experimental_design": {
            "overall_design": _join(metadata.get("overall_design")),
            "experimental_type": _join(metadata.get("type")),
            "tissue": samples[0].get("tissue") if samples else None,
        },
        "sequencing_technology": {
            "platform_id": record.get("platform_id") or _first(sample_meta.get("platform_id")),
            "instrument_model": (samples[0].get("instrument_model") if samples else None)
                or _first(sample_meta.get("instrument_model")),
            "library_strategy": (samples[0].get("library_strategy") if samples else None)
                or _first(sample_meta.get("library_strategy")),
            "library_source": _join(sample_meta.get("library_source")),
            "library_selection": _join(sample_meta.get("library_selection")),
            "molecule": _join(sample_meta.get("molecule_ch1")),
            "channel_count": _first(sample_meta.get("channel_count")),
        },
        "experimental_protocols": {
            "growth_protocol": _join(sample_meta.get("growth_protocol_ch1")),
            "treatment_protocol": _join(sample_meta.get("treatment_protocol_ch1")),
            "extraction_protocol": sample_meta.get("extract_protocol_ch1"),
        },
        "data_processing": sample_meta.get("data_processing"),
        "study_metadata": {
            "status": _first(metadata.get("status")),
            "submission_date": _first(metadata.get("submission_date")),
            "last_update_date": _first(metadata.get("last_update_date")),
            "bioproject": metadata.get("bioproject"),
        },
        "series_relationships": {
            "is_superseries": bool(metadata.get("subseries_of_this_superseries")),
            "subseries_ids": metadata.get("subseries_of_this_superseries"),
            "is_subseries": bool(metadata.get("parent_series_id")),
            "parent_series_id": metadata.get("parent_series_id"),
        },
    }
    return _clean(study_json)


def _parse_characteristics(
    characteristics: Optional[List[str]],
) -> Tuple[Dict[str, str], Dict[str, str], Dict[str, str]]:
    """Split a GEO ``characteristics_ch1`` list into biological/condition/other buckets.

    Uses exact (whole-key) matching against known field-name sets — never a
    substring match — so e.g. a "developmental stage" key can never be
    mis-classified as "age".

    Args:
        characteristics: A list of ``"key: value"`` strings from GEO sample metadata.

    Returns:
        A ``(biological, experimental_conditions, other)`` tuple of dicts.
    """
    biological: Dict[str, str] = {}
    conditions: Dict[str, str] = {}
    other: Dict[str, str] = {}

    for entry in characteristics or []:
        if ":" not in entry:
            continue
        key, value = entry.split(":", 1)
        key_lower = key.strip().lower()
        value = value.strip()

        if key_lower in _BIO_CHARACTERISTIC_KEYS:
            biological[_normalize_key(key_lower)] = value
        elif key_lower in _CONDITION_CHARACTERISTIC_KEYS:
            conditions[_normalize_key(key_lower)] = value
        else:
            other[_normalize_key(key_lower)] = value

    return biological, conditions, other


def _parse_relations(relations: Optional[List[str]]) -> Tuple[Optional[str], Optional[str]]:
    """Extract BioSample and SRA experiment accessions from a GEO ``relation`` list.

    Args:
        relations: A list of GEO ``relation`` metadata strings.

    Returns:
        A ``(biosample_accession, sra_experiment_accession)`` tuple; either may be ``None``.
    """
    biosample = sra_experiment = None
    for relation in relations or []:
        if "SAMN" in relation:
            biosample = relation.split(":")[-1].strip()
        if "SRX" in relation:
            sra_experiment = relation.split(":")[-1].strip()
    return biosample, sra_experiment


def generate_sample_semantic(record: Dict[str, Any]) -> Dict[str, Any]:
    """Convert a GSE database record into a clean, semantic sample-level document.

    Each sample is built directly from its own complete stored metadata — there
    is no common/specific split and no merge-back step, so every sample is
    always fully self-contained and nothing can be silently lost regardless of
    how many samples the GSE has (including the single-sample case).

    ``technical_metadata`` is deliberately trimmed to only the fields useful
    for sample-level retrieval/filtering (platform_id, instrument_model,
    library_strategy); fields that are study-wide and already captured once
    in ``generate_study_semantic()`` (library_source, library_selection,
    data_processing) are intentionally *not* repeated here — they're
    identical across every sample in a GSE, so duplicating them per sample
    only adds storage/embedding cost without adding information.

    Args:
        record: A GSE record as returned by :func:`get_gse_record_from_db`.

    Returns:
        A semantic sample JSON document (nulls/empties removed), containing one
        entry per sample under ``"samples"``.
    """
    gse_id = record.get("gse_id")
    raw_samples = record.get("sample_records") or []

    samples = []
    for sample in raw_samples:
        meta = sample.get("extra_metadata", {}) or {}
        biological, conditions, other_chars = _parse_characteristics(meta.get("characteristics_ch1"))
        biosample, sra_experiment = _parse_relations(meta.get("relation"))

        samples.append(
            {
                "gsm_id": sample.get("gsm_id"),
                "title": sample.get("title"),
                "source_name": sample.get("source_name"),
                "biological_metadata": {
                    "tissue": sample.get("tissue") or biological.get("tissue"),
                    "cell_line": biological.get("cell_line"),
                    "cell_type": biological.get("cell_type"),
                    "genotype": biological.get("genotype"),
                    "specimen": biological.get("specimen"),
                    "disease": biological.get("disease"),
                    "organism": sample.get("organism") or biological.get("organism"),
                    "strain": sample.get("strain") or biological.get("strain"),
                },
                "experimental_conditions": {
                    # Note: "age" alone is kept here. A GEO characteristic like
                    # "age: 41 year" was previously being classified twice —
                    # once into the sample's top-level `timepoint` field (fed
                    # into a separate "time" output key) and again into
                    # `conditions["age"]` — producing a duplicate value under
                    # two different keys for the same underlying data.
                    "treatment": conditions.get("treatment"),
                    "age": conditions.get("age"),
                    "stage": conditions.get("stage"),
                    "exposure": conditions.get("exposure"),
                },
                "technical_metadata": {
                    # Only the fields useful for sample-level retrieval are
                    # kept here. library_source, library_selection, and
                    # data_processing are identical across every sample in a
                    # GSE and already captured once in the study JSON's
                    # sequencing_technology/data_processing — duplicating them
                    # per sample adds storage/embedding cost with no new
                    # information.
                    "library_strategy": sample.get("library_strategy") or _first(meta.get("library_strategy")),
                    "instrument_model": sample.get("instrument_model") or _first(meta.get("instrument_model")),
                    "platform_id": sample.get("platform_id"),
                },
                "descriptions": meta.get("description"),
                "accessions": {"biosample": biosample, "sra_experiment": sra_experiment},
                "raw_extra_metadata": other_chars,
            }
        )

    return _clean(
        {
            "gse_id": gse_id,
            "sample_summary": {"total_samples": len(samples)},
            "samples": samples,
        }
    )


def normalize_gse(gse_id: str, cfg: PipelineConfig) -> Tuple[Path, Path]:
    """Normalize a single GSE, read straight from the database, into final JSON files.

    Writes exactly two files: ``{gse_id}.study.json`` and ``{gse_id}.sample.json``,
    both atomically. Updates the registry to ``normalized`` on success, or
    ``failed`` on error (leaving it eligible to retry on a future run since it
    stays picked up by :func:`select_ids_to_normalize` only while ``fetched``).

    Args:
        gse_id: The GSE accession to normalize (must already be ``fetched``).
        cfg: The active pipeline configuration.

    Returns:
        A ``(study_json_path, sample_json_path)`` tuple.

    Raises:
        ValueError: If ``gse_id`` has no stored data (ingest never completed).
    """
    record = get_gse_record_from_db(gse_id, cfg.db_path)
    if record is None:
        raise ValueError(f"GSE not found in database: {gse_id}")

    study = generate_study_semantic(record)
    study_path = cfg.semantic_study_dir / f"{gse_id}.study.json"
    _atomic_write_json(study_path, study)

    sample = generate_sample_semantic(record)
    sample_path = cfg.semantic_sample_dir / f"{gse_id}.sample.json"
    _atomic_write_json(sample_path, sample)

    return study_path, sample_path


def normalize_pending(cfg: PipelineConfig) -> Tuple[int, int]:
    """Normalize every registry row that is ``fetched`` but not ``normalized``.

    Re-scans the whole registry (not just the current run's batch) so any GSE
    left over from a previous crashed/interrupted run is always caught and
    completed — this guarantees no fetched GSE is ever permanently missed.

    Args:
        cfg: The active pipeline configuration.

    Returns:
        A ``(succeeded, failed)`` count tuple.
    """
    pending = select_ids_to_normalize(cfg.db_path)
    if not pending:
        logger.info("No GSEs pending normalization.")
        return 0, 0

    succeeded = failed = 0
    for gse_id in tqdm(pending, desc="Normalizing GSEs"):
        try:
            study_path, sample_path = normalize_gse(gse_id, cfg)
            mark_status(cfg.db_path, gse_id, "normalized", timestamp_column="normalized_at")
            succeeded += 1
            logger.info("Normalized %s -> %s, %s", gse_id, study_path.name, sample_path.name)
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to normalize %s.", gse_id)
            # Distinct from a fetch-stage "failed": the GSE was already
            # ingested successfully, so a retry should only re-normalize,
            # never re-download (see select_ids_to_fetch vs select_ids_to_normalize).
            mark_status(cfg.db_path, gse_id, "normalize_failed", error=f"normalize: {exc}")
            failed += 1

    logger.info("Normalization complete: %d succeeded, %d failed (of %d).", succeeded, failed, len(pending))
    return succeeded, failed


# %% [markdown]
# ## Orchestration
#
# Discover -> register -> ingest a bounded batch -> normalize every pending GSE
# (including any left over from earlier runs). Every step is registry-driven and
# safe to re-run.

# %%
def run_pipeline(cfg: Optional[PipelineConfig] = None) -> Dict[str, Any]:
    """Run the full GEO metadata pipeline for one bounded, resumable batch.

    Args:
        cfg: Pipeline configuration. Defaults to ``PipelineConfig()`` (a
            5-year rolling window, 50-GSE fetch limit) if not provided.

    Returns:
        A summary dict with discovery/ingest/normalize counts and the final
        registry status breakdown.
    """
    cfg = cfg or PipelineConfig()
    cfg.ensure_dirs()
    init_db(cfg.db_path)

    if cfg.skip_discovery:
        logger.info("skip_discovery=True — reusing the existing registry, no NCBI discovery call made.")
        discovered_ids: List[str] = []
        newly_registered = 0
    else:
        discovered_ids = discover_gse_ids(cfg)
        newly_registered = register_discovered_ids(discovered_ids, cfg.db_path)

    to_fetch = select_ids_to_fetch(cfg.db_path, cfg.fetch_limit)
    if to_fetch:
        ingested_ok = ingested_failed = 0
        for i in range(0, len(to_fetch), cfg.batch_size):
            chunk = to_fetch[i : i + cfg.batch_size]
            ok, fail = ingest_gse_batch(chunk, cfg)
            ingested_ok += ok
            ingested_failed += fail
    else:
        logger.info("Nothing new to fetch this run.")
        ingested_ok = ingested_failed = 0

    normalized_ok, normalized_failed = normalize_pending(cfg)

    summary = {
        "discovered_this_run": len(discovered_ids),
        "newly_registered": newly_registered,
        "ingested_ok": ingested_ok,
        "ingested_failed": ingested_failed,
        "normalized_ok": normalized_ok,
        "normalized_failed": normalized_failed,
        "registry_status": registry_summary(cfg.db_path),
        "study_dir": str(cfg.semantic_study_dir),
        "sample_dir": str(cfg.semantic_sample_dir),
    }
    logger.info("Pipeline run complete: %s", json.dumps(summary, indent=2))
    return summary


# %% [markdown]
# ## Inspection — preview the most recently fetched GSEs
#
# Prints the study/sample JSON for the N most recently *fetched* GSEs (by
# ``fetched_at`` in the registry — this reflects when a GSE was ingested,
# regardless of whether it has been normalized yet). Handy for a quick sanity
# check right after a run without hunting through the output folders by hand.

# %%
def print_recent_fetched(cfg: PipelineConfig, n: int = 5, kind: str = "both") -> List[str]:
    """Print the study/sample JSON for the N most recently fetched GSEs.

    Args:
        cfg: The active pipeline configuration.
        n: How many recent GSEs to show (tune as needed).
        kind: ``"study"``, ``"sample"``, or ``"both"`` — which file(s) to print.

    Returns:
        The list of GSE accessions that were shown, most recent first.
    """
    with db_connection(cfg.db_path) as conn:
        rows = conn.execute(
            """
            SELECT gse_id, fetched_at FROM gse_registry
            WHERE fetched_at IS NOT NULL
            ORDER BY fetched_at DESC
            LIMIT ?;
            """,
            (n,),
        ).fetchall()

    if not rows:
        print("No fetched GSEs yet.")
        return []

    gse_ids = []
    for gse_id, fetched_at in rows:
        gse_ids.append(gse_id)
        print(f"\n{'=' * 70}\n{gse_id}  (fetched at {fetched_at})\n{'=' * 70}")

        if kind in ("study", "both"):
            study_path = cfg.semantic_study_dir / f"{gse_id}.study.json"
            if study_path.exists():
                print(f"--- {study_path.name} ---")
                print(study_path.read_text(encoding="utf-8"))
            else:
                print(f"({study_path.name} not written yet — not normalized)")

        if kind in ("sample", "both"):
            sample_path = cfg.semantic_sample_dir / f"{gse_id}.sample.json"
            if sample_path.exists():
                print(f"--- {sample_path.name} ---")
                print(sample_path.read_text(encoding="utf-8"))
            else:
                print(f"({sample_path.name} not written yet — not normalized)")

    return gse_ids


def print_recent_fetched(cfg: PipelineConfig, n: int = 5, kind: str = "both") -> List[str]:
    """Print the study/sample JSON for the N most recently fetched GSEs.

    Args:
        cfg: The active pipeline configuration.
        n: How many recent GSEs to show (tune as needed).
        kind: ``"study"``, ``"sample"``, or ``"both"`` — which file(s) to print.

    Returns:
        The list of GSE accessions that were shown, most recent first.
    """
    with db_connection(cfg.db_path) as conn:
        rows = conn.execute(
            """
            SELECT gse_id, fetched_at FROM gse_registry
            WHERE fetched_at IS NOT NULL
            ORDER BY fetched_at DESC
            LIMIT ?;
            """,
            (n,),
        ).fetchall()

    if not rows:
        print("No fetched GSEs yet.")
        return []

    gse_ids = []
    for gse_id, fetched_at in rows:
        gse_ids.append(gse_id)
        print(f"\n{'=' * 70}\n{gse_id}  (fetched at {fetched_at})\n{'=' * 70}")

        if kind in ("study", "both"):
            study_path = cfg.semantic_study_dir / f"{gse_id}.study.json"
            if study_path.exists():
                print(f"--- {study_path.name} ---")
                print(study_path.read_text(encoding="utf-8"))
            else:
                print(f"({study_path.name} not written yet — not normalized)")

        if kind in ("sample", "both"):
            sample_path = cfg.semantic_sample_dir / f"{gse_id}.sample.json"
            if sample_path.exists():
                print(f"--- {sample_path.name} ---")
                print(sample_path.read_text(encoding="utf-8"))
            else:
                print(f"({sample_path.name} not written yet — not normalized)")

    return gse_ids


# %% [markdown]
# ## Inspection — find normalized GSEs with zero samples
#
# Scans every ``{gse_id}.sample.json`` in ``semantic_sample_dir`` and reports
# any with ``sample_summary.total_samples == 0``. This is the expected
# signature of an empty **SuperSeries** (see ``series_relationships`` in the
# matching ``.study.json`` — a SuperSeries carries no sample data of its own,
# only links to its SubSeries), but it can also flag a genuine ingestion/
# parsing problem worth a closer look, so both cases are worth inspecting.

# %%
def find_zero_sample_gses(cfg: PipelineConfig, verbose: bool = True) -> List[str]:
    """Find every normalized GSE whose sample.json has zero samples.

    Args:
        cfg: The active pipeline configuration.
        verbose: If True, print each GSE ID (and whether it's a SuperSeries,
            per its study.json, when that can be determined) as it's found.

    Returns:
        A list of GSE accessions with ``total_samples == 0``.
    """
    empty_gses: List[str] = []

    for sample_path in sorted(cfg.semantic_sample_dir.glob("*.sample.json")):
        try:
            data = json.loads(sample_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as exc:
            logger.warning("Could not read %s: %s", sample_path.name, exc)
            continue

        total_samples = data.get("sample_summary", {}).get("total_samples", 0)
        if total_samples != 0:
            continue

        gse_id = data.get("gse_id", sample_path.stem.replace(".sample", ""))
        empty_gses.append(gse_id)

        if verbose:
            is_superseries = None
            study_path = cfg.semantic_study_dir / f"{gse_id}.study.json"
            if study_path.exists():
                try:
                    study_data = json.loads(study_path.read_text(encoding="utf-8"))
                    is_superseries = study_data.get("series_relationships", {}).get("is_superseries")
                except (json.JSONDecodeError, OSError):
                    pass
            tag = "SuperSeries (expected)" if is_superseries else "NOT a SuperSeries — worth checking"
            print(f"{gse_id}  [{tag}]")

    if verbose:
        print(f"\nTotal GSE(s) with 0 samples: {len(empty_gses)}")

    return empty_gses


# %% input parameters - default 5 years.
# year=2026, month=6, fetch_limit=20
# start_date="2026/01/15", end_date="2026/03/20", fetch_limit=50
# years_back=2, fetch_limit=30
# fetch_limit=50,batch_size=25 - default
# skip_discovery=True,   # <-- NCBI discovery call off, only existing registry use.

if __name__ == "__main__":
    # Example: last full month, 20 new GSEs per run, default batch_size (25).
    config = PipelineConfig(skip_discovery=True, fetch_limit=200, batch_size=25)
    run_pipeline(config)


# Run the inspection cells
# %%
if __name__ == "__main__":
    # Tune n as needed; kind="study", "sample", or "both".
    print_recent_fetched(config, n=5, kind="sample")

# %%
if __name__ == "__main__":
    zero_sample_gses = find_zero_sample_gses(config)

2026-08-16 21:00:32,121 [INFO] geo_pipeline: Database ready at geo_work\geo_data.db
2026-08-16 21:00:32,123 [INFO] geo_pipeline: skip_discovery=True — reusing the existing registry, no NCBI discovery call made.
Ingesting GSE batch:   4%|▍         | 1/25 [00:01<00:45,  1.89s/it]2026-08-16 21:00:34,071 [INFO] geo_pipeline: Ingested GSE294705: 94 sample(s).
2026-08-16 21:00:34,430 [INFO] geo_pipeline: Ingested GSE306118: 4 sample(s).
Ingesting GSE batch:  16%|█▌        | 4/25 [00:03<00:17,  1.20it/s]2026-08-16 21:00:35,672 [INFO] geo_pipeline: Ingested GSE289185: 29 sample(s).
2026-08-16 21:00:52,292 [INFO] geo_pipeline: Ingested GSE286432: 21 sample(s).
Ingesting GSE batch:  24%|██▍       | 6/25 [00:20<01:26,  4.55s/it]2026-08-16 21:00:52,307 [INFO] geo_pipeline: Ingested GSE286320: 1 sample(s).
2026-08-16 21:00:53,942 [INFO] geo_pipeline: Ingested GSE286319: 4 sample(s).
Ingesting GSE batch:  32%|███▏      | 8/25 [00:21<00:51,  3.01s/it]2026-08-16 21:00:54,030 [INFO] geo_pipeline: Inges


GSE318420  (fetched at 2026-08-16 21:09:41)
--- GSE318420.sample.json ---
{
  "gse_id": "GSE318420",
  "sample_summary": {
    "total_samples": 210
  },
  "samples": [
    {
      "gsm_id": "GSM9371441",
      "title": "HCC PBMC patient A-07 timepoint C2D1 scRNA-seq replicate 1",
      "source_name": "Hepatocellular Carcinoma (HCC)",
      "biological_metadata": {
        "tissue": "Hepatocellular Carcinoma (HCC)",
        "cell_type": "peripheral blood mononuclear cells (PBMCs)",
        "genotype": "A-07",
        "organism": "Homo sapiens"
      },
      "experimental_conditions": {
        "treatment": "nivolumab"
      },
      "technical_metadata": {
        "library_strategy": "RNA-Seq",
        "instrument_model": "Illumina NovaSeq 6000",
        "platform_id": "GPL24676"
      },
      "descriptions": [
        "Library name: A-07-01_GEX"
      ],
      "accessions": {
        "biosample": "//www.ncbi.nlm.nih.gov/biosample/SAMN53912211",
        "sra_experiment": "//www.ncbi.

# GEO DB Explorer


In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "geo_work/geo_data.db"  

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

tables = [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print(f"Connected: {DB_PATH}")
print(f"Tables: {tables}")


## Pipeline health snapshot
Quick view of overall pipeline state — how much is left to process, fail rate, etc.

In [ ]:
total = cur.execute("SELECT COUNT(*) FROM gse_registry").fetchone()[0]
normalized = cur.execute("SELECT COUNT(*) FROM gse_registry WHERE status='normalized'").fetchone()[0]
failed = cur.execute("SELECT COUNT(*) FROM gse_registry WHERE status='failed'").fetchone()[0]
discovered = cur.execute("SELECT COUNT(*) FROM gse_registry WHERE status='discovered'").fetchone()[0]

print(f"Total        : {total}")
print(f"Normalized   : {normalized} ({normalized/total:.1%})")
print(f"Failed       : {failed} ({failed/total:.1%})")
print(f"Pending      : {discovered} ({discovered/total:.1%})")


## Find empty GSE
Two kinds of "empty" checked here:
- **No sample**: present in `gse_data` but `sample_records` is an empty `[]`
- **Blank record**: title/organism/summary are all empty (broken/corrupt fetch)

In [ ]:
rows = cur.execute("SELECT gse_id, title, organism, summary, sample_records FROM gse_data").fetchall()

no_sample, blank_record = [], []

for r in rows:
    try:
        samples = json.loads(r["sample_records"]) if r["sample_records"] else []
    except json.JSONDecodeError:
        samples = None

    if not samples:
        no_sample.append(r["gse_id"])

    if not any([r["title"], r["organism"], r["summary"]]):
        blank_record.append(r["gse_id"])

print(f"No-sample GSE ({len(no_sample)}):")
for g in no_sample:
    print(" -", g)

print(f"\nBlank record GSE ({len(blank_record)}):")
for g in blank_record:
    print(" -", g)
